# Quick demo: classify one e-puck camera image with my CNN

1. **Runtime → Change runtime type → T4 GPU**
2. Run **cell 1** (gets the model: loads it from Google Drive if I saved it before, otherwise trains it in ~2–3 min on the T4 and saves it)
3. Run **cell 2**, click *Choose files* and pick **one** image from the e-puck camera. It prints the CNN's guess.

Same CNN as `ARAIP_CIFAR10_CNN.ipynb` (3 VGG-style blocks, 306,154 parameters), trained for fewer epochs so it's ready quickly.

In [ ]:
# CELL 1: get the model
import os, io, time, numpy as np, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
CLASSES = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
print("GPU:", tf.config.list_physical_devices("GPU") or "NONE - switch to T4 GPU!")

MODEL_PATH = "araip_cnn_demo.keras"
USE_DRIVE = False   # True = keep the trained model in Google Drive so I only train once (asks to sign in)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/ARAIP", exist_ok=True)
    MODEL_PATH = "/content/drive/MyDrive/ARAIP/araip_cnn_demo.keras"

def build_cnn():
    inp = keras.Input(shape=(32, 32, 3))
    x = inp
    for f, d in [(32, .2), (64, .3), (128, .4)]:
        for _ in range(2):
            x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x); x = layers.Dropout(d)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x); x = layers.Dropout(.4)(x)
    return keras.Model(inp, layers.Dense(10, activation="softmax")(x))

if os.path.exists(MODEL_PATH):
    model = keras.models.load_model(MODEL_PATH)
    print("Loaded my saved model from", MODEL_PATH)
else:
    t0 = time.time()
    import pandas as pd
    from PIL import Image as _I
    base = "https://huggingface.co/datasets/uoft-cs/cifar10/resolve/main/plain_text/"   # fast mirror of CIFAR-10
    def part(n):
        df = pd.read_parquet(base + n)
        return np.stack([np.array(_I.open(io.BytesIO(d["bytes"])).convert("RGB")) for d in df["img"]]), df["label"].to_numpy()
    xtr, ytr = part("train-00000-of-00001.parquet"); xte, yte = part("test-00000-of-00001.parquet")
    xtr, xte = xtr.astype("float32") / 255, xte.astype("float32") / 255
    print(f"data loaded in {time.time()-t0:.0f} s, training...")
    aug = keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomTranslation(.1, .1, fill_mode="reflect")])
    ds = (tf.data.Dataset.from_tensor_slices((xtr, ytr)).shuffle(10000).batch(128)
          .map(lambda x, y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE))
    model = build_cnn()
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    EPOCHS = 15
    sched = keras.callbacks.LearningRateScheduler(lambda e: 1e-3 if e < 10 else 3e-4)
    model.fit(ds, epochs=EPOCHS, callbacks=[sched], verbose=2)
    acc = model.evaluate(xte, yte, verbose=0)[1]
    print(f"Trained in {(time.time()-t0)/60:.1f} min, test accuracy {acc:.1%}")
    model.save(MODEL_PATH); print("Saved to", MODEL_PATH)


In [ ]:
# CELL 2: upload ONE e-puck camera image and classify it
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

up = files.upload()
name = list(up)[0]
img = Image.open(io.BytesIO(up[name])).convert("RGB")
w, h = img.size; s = min(w, h)                       # centre square crop, like the main notebook
small = img.crop(((w-s)//2, (h-s)//2, (w+s)//2, (h+s)//2)).resize((32, 32), Image.LANCZOS)
probs = model.predict(np.asarray(small, "float32")[None] / 255, verbose=0)[0]
top = np.argsort(-probs)[:3]

print(f"{name}:  the CNN thinks this is a  >>> {CLASSES[top[0]].upper()} <<<  ({probs[top[0]]:.0%})")
for k in top: print(f"   {CLASSES[k]:<11}{probs[k]:6.1%}")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].imshow(img); ax[0].set_title("e-puck camera image"); ax[0].axis("off")
ax[1].imshow(small); ax[1].set_title("what the CNN sees (32x32)"); ax[1].axis("off")
ax[2].barh([CLASSES[k] for k in top][::-1], probs[top][::-1], color="#2563eb"); ax[2].set_xlim(0, 1)
ax[2].set_title(f"Prediction: {CLASSES[top[0]]} ({probs[top[0]]:.0%})")
plt.tight_layout(); plt.show()
